In [65]:
import Pkg;
Pkg.add("StatsBase")

using LinearAlgebra
using StatsBase

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [66]:
using Random, StatsBase,SparseArrays

Random.seed!(1234)  # set the global seed

TaskLocalRNG()

In [67]:
function cliqueSampleMultiEdgeSddEff(a::Vector{ComplexF64}, multiplicity::Vector{Int}, i::Int)
    n = length(a)

    # 1. Pre-calculate magnitudes to avoid repeated abs() calls
    # We copy 'a' logic without mutating the input vector permanently
    abs_a = abs.(a)
    val_i = a[i]
    abs_val_i = abs_a[i]

    # 2. Logic for self_loop and initialization
    # Original logic: self_loop = 2*|a[i]| - sum(|a|)
    # Note: sum(abs_a) includes |a[i]|, so we use that.
    total_weight = abs_val_i
    sum_abs_all = sum(abs_a)
    self_loop = abs(2 * abs_val_i - sum_abs_all)
    #println(self_loop)
    # 3. Initialize Output Matrices
    cliqueSampleLap = zeros(ComplexF64, n, n)
    new_multiedges = zeros(Int, n, n)

    # 4. Handle Diagonal Initialization (Vectorized)
    # Corresponds to: cliqueSampleLap[diagind] .+= self_loop/total_weight * abs.(a)
    # In original code, a[i] was set to 0 before this, so index i gets 0 update.
    ratio = self_loop / total_weight
    @inbounds for k in 1:n
        if k != i
            cliqueSampleLap[k, k] = ratio * abs_a[k]
        end
    end

    # 5. Setup Sampling ONCE (Big Performance Win)
    # We mask position i to 0.0 so we never sample it (removes need for k != i check)
    sampling_weights = copy(abs_a)
    sampling_weights[i] = self_loop
    wv = Weights(sampling_weights) # O(N) operation happens only once now

    # Identify neighbors (indices where weight > threshold)
    # We use valid indices from abs_a (excluding i effectively because we skip it in loop or weights)
    nbrs = findall(x -> x > 1e-12, sampling_weights)

    # 6. Main Loop
    for j in nbrs
        # Pre-calculate j-specific constants
        val_j = a[j]
        abs_val_j = abs_a[j]
        sign_j = sign(a[j]) #val_j / abs_val_j # Equivalent to sign(a[j]) for non-zero
        mult_j = multiplicity[j]

        # Scaling factor from logic: multiWeightJ = a[j] / multiplicity[j]
        # We need this for the wjk calc
        val_j_scaled = val_j / mult_j

        for rep in 1:mult_j
            # O(1) Sampling
            k = sample(wv)

            # We guaranteed k != i via weights. We only check k != j.
            if (k != j) & (j != i) & (k != i)
                val_k = a[k]
                abs_val_k = abs_a[k]

                # Calculate Weight wjk
                # wjk = abs(multiWeightJ * a[k]) / (|a[j]| + |a[k]|)
                wjk = abs(val_j_scaled * val_k) / (abs_val_j + abs_val_k)

                # Update Edge Counts
                @inbounds new_multiedges[j, k] += 1
                @inbounds new_multiedges[k, j] += 1

                # 7. Rank-1 Matrix Update (Scalar optimization)
                # Instead of L += w * b * b', we update 4 indices manually.
                # b[j] = 1, b[k] = -sign(a[k])/sign(a[j])
                s = -(val_k / abs_val_k) / sign_j # s is the value at b[k]

                # The outer product adds:
                # [j,j] -> |1|^2 * w      [j,k] -> 1 * conj(s) * w
                # [k,j] -> s * 1 * w      [k,k] -> |s|^2 * w

                # Note: |s| = 1 for complex sign, so |s|^2 = 1.

                @inbounds begin
                    cliqueSampleLap[j, j] += wjk
                    cliqueSampleLap[k, k] += wjk
                    cliqueSampleLap[j, k] += wjk * conj(s)
                    cliqueSampleLap[k, j] += wjk * s
                end
            end
        end
    end

    return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdgeSddEff (generic function with 1 method)

In [68]:
function initMultiedges(L, nr_multi)
  n = size(L)[1]
  multiedges = zeros(Int, n,n,)
  A_mask = abs.(-L + Diagonal(diag(L))).> 0
  multiedges[A_mask] .= nr_multi
  return multiedges
end

initMultiedges (generic function with 1 method)

In [69]:
# This is still the same algorithm and some index handling
function apxCholMultiSDD(L, nr_multi)
  multi_edges = initMultiedges(L, nr_multi)
  n = size(L)[1]
  S = copy(L)
  fac = zeros(ComplexF64,n,n)
  for i = 1:(n-1)

    li = S[:,i]/S[i,i]^(1/2)
    fac[:,i] = li

    ai = S[:,i]
    di = S[i,i]

    # S without star on i
    S_noi = copy(S)
    S_noi -=Diagonal(abs.(ai))
    S_noi[:,i] .= 0
    S_noi[i,:] .= 0
    sampled_clique, new_multiedges = cliqueSampleMultiEdgeSddEff(ai, multi_edges[:,i],i)
    S = S_noi + sampled_clique
    multi_edges += new_multiedges
    if i==n-1
      fac[n,n] = sqrt(abs(S_noi[n,n]+ sampled_clique[n,n]))
    end
  end
  return fac
end

apxCholMultiSDD (generic function with 1 method)

In [70]:
function generate_hermitian_dominant(n::Int; strict::Bool=true)
    # 1. Generate a random complex matrix
    A = rand(ComplexF64, n, n)

    # 2. Make it Hermitian: H = A + A'
    # Note: The diagonal of (A + A') is guaranteed to be real because z + conj(z) = 2*Real(z)
    H = A + A'

    # 3. Adjust Diagonal to enforce Dominance
    for i in 1:n
        # Calculate sum of absolute values of off-diagonal elements in row i
        off_diag_sum = sum(abs, H[i, :]) - abs(H[i, i])

        # Determine the buffer to add (random positive value for strictness)
        buffer = strict ? (0.1 + rand()) : 0.0

        # Set the diagonal element. It must be real to remain Hermitian.
        H[i, i] = off_diag_sum + buffer
    end

    return H
end

generate_hermitian_dominant (generic function with 1 method)

In [72]:
for i in 1:10
  n = 100
  S = generate_hermitian_dominant(n)
  fApxChol = apxCholMultiSDD(S, 100) # Second param specifies the number of multiedges
  rel_eigs = eigvals(S-fApxChol*fApxChol')

  # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / maximum(abs.(eigvals(S))))
end

0.005378273560395605
0.004597402725941163
0.0047873043434361945
0.004176795901341596
0.004231777736101578
0.004609540605872456
0.004423912520529985
0.0047671800150192595
0.004281418566739049
0.00474394580054363


In [73]:
function cliqueSampleMultiEdgeSddEffSparse(a::SparseVector{ComplexF64, Int}, multiplicity::SparseVector{Int, Int}, i::Int)
    n = length(a)

    # 1. Work with stored values directly (O(degree) instead of O(N))
    # 'rows' contains the indices of neighbors (and likely 'i' itself)
    rows = a.nzind
    vals = a.nzval

    # Pre-calculate magnitudes for stored entries only
    # This creates a small vector of size nnz(a), not N
    abs_vals = abs.(vals)

    # 2. Find index of 'i' in the sparse storage
    # SparseVectors are sorted, so we can use searchsortedfirst
    ptr_i = searchsortedfirst(rows, i)

    # Check if 'i' is actually stored in the sparse vector
    has_i = (ptr_i <= length(rows)) && (rows[ptr_i] == i)

    val_i = has_i ? vals[ptr_i] : 0.0 + 0.0im
    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0

    # 3. Logic for self_loop
    # sum(abs_vals) sums only the stored non-zeros
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    # 4. Initialize Output Sparse Matrices
    # We calculate capacity to avoid resizing
    # Note: accessing sparse multiplicity[row] is O(log nnz), which is acceptable
    total_reps = 0
    @inbounds for (idx, r) in enumerate(rows)
        if r != i
            # If multiplicity is sparse, ensure we handle missing keys (assumed 0 or 1 depending on logic)
            # Assuming multiplicity is consistent with 'a' structure:
            total_reps += multiplicity[r]
        end
    end

    # Pre-allocate output triplets
    I_lap = Int[]; sizehint!(I_lap, n + 4*total_reps)
    J_lap = Int[]; sizehint!(J_lap, n + 4*total_reps)
    V_lap = ComplexF64[]; sizehint!(V_lap, n + 4*total_reps)

    I_edge = Int[]; sizehint!(I_edge, 2*total_reps)
    J_edge = Int[]; sizehint!(J_edge, 2*total_reps)
    V_edge = Int[]; sizehint!(V_edge, 2*total_reps)

    # 5. Handle Diagonal Initialization
    # We only iterate over existing non-zeros (neighbors)
    total_weight = abs_val_i
    ratio = (total_weight > 1e-12) ? self_loop / total_weight : 0.0

    @inbounds for (idx, k) in enumerate(rows)
        if k != i
            push!(I_lap, k); push!(J_lap, k); push!(V_lap, ratio * abs_vals[idx])
        end
    end

    # 6. Setup Compressed Sampling
    # We create weights for the 'rows' array, not the full 1:n range.
    sampling_weights = copy(abs_vals)
    if has_i
        sampling_weights[ptr_i] = self_loop
    else
        # If 'i' wasn't in sparse structure, we technically can't sample it easily
        # without resizing arrays. But usually for Laplacians, diagonal 'i' is always present.
        # If strictly necessary, one would push!(rows, i) and push!(sampling_weights, self_loop), re-sort.
        # Assuming 'i' exists in 'a' for SDD matrices:
        error("Diagonal entry a[i] is missing from sparse structure")
    end

    wv = Weights(sampling_weights) # Size is degree(i), very small

    # 7. Main Loop
    # Iterate over indices into the compressed vectors 'rows'/'vals'
    for (j_idx, j) in enumerate(rows)
        # Skip if weight is negligible or it is 'i'
        if j == i || sampling_weights[j_idx] <= 1e-12
            continue
        end

        val_j = vals[j_idx]
        abs_val_j = abs_vals[j_idx]
        inv_sign_j = conj(val_j / abs_val_j)

        mult_j = multiplicity[j] # Sparse lookup
        val_j_scaled = val_j / mult_j

        for rep in 1:mult_j
            # O(1) Sampling from compressed vector
            # This returns an index into 'rows', not the node index directly
            k_idx = sample(wv)
            k = rows[k_idx]

            if k != j && k != i
                val_k = vals[k_idx]
                abs_val_k = abs_vals[k_idx]

                # Weight calculation
                wjk = abs(val_j_scaled * val_k) / (abs_val_j + abs_val_k)

                # Update Edges
                push!(I_edge, j); push!(J_edge, k); push!(V_edge, 1)
                push!(I_edge, k); push!(J_edge, j); push!(V_edge, 1)

                # Update Laplacian (Rank-1)
                s = -(val_k / abs_val_k) * inv_sign_j

                push!(I_lap, j); push!(J_lap, j); push!(V_lap, wjk)
                push!(I_lap, k); push!(J_lap, k); push!(V_lap, wjk)
                push!(I_lap, j); push!(J_lap, k); push!(V_lap, wjk * conj(s))
                push!(I_lap, k); push!(J_lap, j); push!(V_lap, wjk * s)
            end
        end
    end

    # Construct final matrices
    cliqueSampleLap = sparse(I_lap, J_lap, V_lap, n, n)
    new_multiedges = sparse(I_edge, J_edge, V_edge, n, n)

    return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdgeSddEffSparse (generic function with 1 method)

In [79]:
function cliqueSampleMultiEdgeSddEffSparseOpt(a::SparseVector{ComplexF64, Int}, multiplicity::SparseVector{Int, Int}, i::Int)
    n = length(a)

    # Access raw storage directly
    rows = a.nzind
    vals = a.nzval
    m = length(rows) # Degree of i (number of stored entries)

    # 1. Pre-compute magnitudes and signs (Vectorized)
    # avoiding sqrt and division inside the hot loop
    abs_vals = abs.(vals)
    signs = vals ./ abs_vals

    # 2. Efficiently find index of diagonal 'i'
    ptr_i = searchsortedfirst(rows, i)
    has_i = (ptr_i <= m) && (rows[ptr_i] == i)

    abs_val_i = has_i ? abs_vals[ptr_i] : 0.0

    # 3. Calculate Self Loop
    sum_abs_all = sum(abs_vals)
    self_loop = abs(2 * abs_val_i - sum_abs_all)

    # 4. Prepare Sampling Weights
    # We copy abs_vals because we need to modify the weight for 'i'
    sampling_weights = copy(abs_vals)
    if has_i
        sampling_weights[ptr_i] = self_loop
    else
        error("Diagonal entry a[i] is missing from sparse structure")
    end

    wv = Weights(sampling_weights)

    # 5. Fast Multiplicity Lookup
    # Extract multiplicities into a dense vector aligned with 'rows'.
    # This prevents O(log N) lookups inside the hot loop.
    node_mults = Vector{Int}(undef, m)
    total_reps = 0

    @inbounds for k in 1:m
        if k == ptr_i
            node_mults[k] = 0
        else
            # Accessing sparse vector here is okay as it is done only 'm' times total
            mu = multiplicity[rows[k]]
            node_mults[k] = mu
            total_reps += mu
        end
    end

    # 6. Pre-allocate Output Arrays (Exact or Upper Bound)
    # Laplacian: (neighbors) diagonal entries + 4 * total_reps (rank-1 updates)
    # Edge: 2 * total_reps
    num_neighbors = has_i ? m - 1 : m
    max_lap_entries = num_neighbors + 4 * total_reps
    max_edge_entries = 2 * total_reps

    I_lap = Vector{Int}(undef, max_lap_entries)
    J_lap = Vector{Int}(undef, max_lap_entries)
    V_lap = Vector{ComplexF64}(undef, max_lap_entries)

    I_edge = Vector{Int}(undef, max_edge_entries)
    J_edge = Vector{Int}(undef, max_edge_entries)
    V_edge = Vector{Int}(undef, max_edge_entries)

    # 7. Diagonal Initialization
    idx_lap = 1
    total_weight = abs_val_i
    ratio = (total_weight > 1e-12) ? self_loop / total_weight : 0.0

    @inbounds for k in 1:m
        if k != ptr_i
            I_lap[idx_lap] = rows[k]
            J_lap[idx_lap] = rows[k]
            V_lap[idx_lap] = ratio * abs_vals[k]
            idx_lap += 1
        end
    end

    # 8. Main Loop
    idx_edge = 1

    @inbounds for j_idx in 1:m
        # Skip 'i' or negligible weights
        if j_idx == ptr_i || sampling_weights[j_idx] <= 1e-12
            continue
        end

        # Load j properties from pre-computed vectors
        row_j = rows[j_idx]
        val_j = vals[j_idx]
        abs_val_j = abs_vals[j_idx]
        inv_sign_j = conj(signs[j_idx])

        mult_j = node_mults[j_idx]
        val_j_scaled = val_j / mult_j

        for rep in 1:mult_j
            # Sample k index from compressed list
            k_idx = sample(wv)

            # Skip invalid samples
            if k_idx != j_idx && k_idx != ptr_i
                row_k = rows[k_idx]
                val_k = vals[k_idx]
                abs_val_k = abs_vals[k_idx]

                # Weight Calculation
                wjk = abs(val_j_scaled * val_k) / (abs_val_j + abs_val_k)

                # Laplacian Rank-1 Update components
                # s = -(val_k / abs_val_k) * inv_sign_j
                s = -(signs[k_idx]) * inv_sign_j

                val_diag = wjk
                val_off1 = wjk * conj(s)
                val_off2 = wjk * s

                # Fill Edge Buffers
                I_edge[idx_edge] = row_j; J_edge[idx_edge] = row_k; V_edge[idx_edge] = 1
                idx_edge += 1
                I_edge[idx_edge] = row_k; J_edge[idx_edge] = row_j; V_edge[idx_edge] = 1
                idx_edge += 1

                # Fill Laplacian Buffers
                I_lap[idx_lap] = row_j; J_lap[idx_lap] = row_j; V_lap[idx_lap] = val_diag
                idx_lap += 1
                I_lap[idx_lap] = row_k; J_lap[idx_lap] = row_k; V_lap[idx_lap] = val_diag
                idx_lap += 1
                I_lap[idx_lap] = row_j; J_lap[idx_lap] = row_k; V_lap[idx_lap] = val_off1
                idx_lap += 1
                I_lap[idx_lap] = row_k; J_lap[idx_lap] = row_j; V_lap[idx_lap] = val_off2
                idx_lap += 1
            end
        end
    end

    # 9. Trim arrays
    # Since we skipped some samples (k == j || k == i), we might have unused space.
    # resize! is cheap (just updates length property).
    resize!(I_lap, idx_lap - 1)
    resize!(J_lap, idx_lap - 1)
    resize!(V_lap, idx_lap - 1)

    resize!(I_edge, idx_edge - 1)
    resize!(J_edge, idx_edge - 1)
    resize!(V_edge, idx_edge - 1)

    # 10. Construct Final Matrices
    cliqueSampleLap = sparse(I_lap, J_lap, V_lap, n, n)
    new_multiedges = sparse(I_edge, J_edge, V_edge, n, n)

    return cliqueSampleLap, new_multiedges
end

cliqueSampleMultiEdgeSddEffSparseOpt (generic function with 1 method)

In [74]:
function initMultiedgesSparse(L::SparseMatrixCSC, nr_multi::Int)
    n = size(L, 1)

    # 1. Extract structural non-zeros from L
    # findnz returns the Row indices (I), Column indices (J), and Values (V)
    rows, cols, vals = findnz(L)

    # 2. Filter out the diagonal entries
    # We want indices where row != col AND the value is essentially non-zero
    # (Though typically structural non-zeros in L are actual edges)
    mask = (rows .!= cols) .& (abs.(vals) .> 1e-12)

    edge_rows = rows[mask]
    edge_cols = cols[mask]

    # 3. Create the values vector
    # Every valid edge gets the value 'nr_multi'
    edge_vals = fill(nr_multi, length(edge_rows))

    # 4. Construct the sparse matrix
    # sparse(I, J, V, m, n) creates the matrix efficiently
    multiedges = sparse(edge_rows, edge_cols, edge_vals, n, n)

    return multiedges
end

initMultiedgesSparse (generic function with 1 method)

In [80]:
function apxCholMultiSDDSparse(L::SparseMatrixCSC, nr_multi::Int)
    n = size(L, 1)

    # 1. Initialize Sparse Structures
    multi_edges = initMultiedgesSparse(L, nr_multi)

    S = copy(L) # Initial sparse copy

    # Initialize triplets for the factor L
    fac_I = Int[]
    fac_J = Int[]
    fac_V = ComplexF64[] # Ensure this is Complex

    sizehint!(fac_I, nnz(L))
    sizehint!(fac_J, nnz(L))
    sizehint!(fac_V, nnz(L))

    for i = 1:(n-1)
        # 2. Extract pivot column
        ai = S[:, i]
        di = S[i, i]

        # In a valid Hermitian Cholesky, diagonal pivots must be real and positive.
        # We take real(di) to ensure type stability and avoid numerical noise (e.g. 1e-18im).
        inv_sqrt_di = 1 / sqrt(real(di))

        # Get nonzero indices and values for column i
        rows = rowvals(S)[nzrange(S, i)]
        vals = nonzeros(S)[nzrange(S, i)]

        for (idx, row) in enumerate(rows)
            # Factor L calculation: L_ki = S_ki / sqrt(S_ii)
            # vals[idx] is S_row,i
            val = vals[idx] * inv_sqrt_di

            push!(fac_I, row)
            push!(fac_J, i)
            push!(fac_V, val)
        end

        # 3. Construct the "Removal Matrix" (M_rem)
        rem_I = Int[]
        rem_J = Int[]
        rem_V = ComplexF64[]

        # Re-iterate the sparse column 'ai' to build the removal updates
        for (idx, row) in enumerate(rows)
            val = vals[idx]

            # 3a. Diagonal Updates
            # For Hermitian SDD, the diagonal dominance is based on magnitude |A_ij|.
            # abs(val) works correctly for Complex numbers (returns modulus).
            push!(rem_I, row); push!(rem_J, row); push!(rem_V, abs(val))

            # 3b. Column i removal (S[row, i])
            # We subtract the value exactly as is
            push!(rem_I, row); push!(rem_J, i); push!(rem_V, val)

            # 3c. Row i removal (S[i, row])
            # HERMITIAN CHANGE: S[i, row] should be conj(S[row, i])
            # Therefore, we must subtract conj(val) to zero it out.
            if row != i
                push!(rem_I, i); push!(rem_J, row); push!(rem_V, conj(val))
            end
        end

        M_rem = sparse(rem_I, rem_J, rem_V, n, n)

        # 4. Sample the Clique
        # Note: Ensure your 'cliqueSample' function returns a Hermitian update
        # (i.e., if it adds edge (u,v) with weight w, it adds (v,u) with weight conj(w)).
        sampled_clique, new_multiedges = cliqueSampleMultiEdgeSddEffSparseOpt(ai, multi_edges[:, i], i)

        # 5. Update Matrices
        S = S - M_rem + sampled_clique
        multi_edges = multi_edges + new_multiedges

        # Handle the final corner case for the last diagonal element
        if i == n - 1
             # S[n,n] is real in Hermitian matrices
             final_val = sqrt(abs(real(S[n, n])))
             push!(fac_I, n); push!(fac_J, n); push!(fac_V, final_val)
        end
    end

    # Construct final sparse factor matrix
    fac = sparse(fac_I, fac_J, fac_V, n, n)
    return fac
end

apxCholMultiSDDSparse (generic function with 1 method)

In [76]:
function generate_random_sparse_hermitian_sdd(n::Int, density::Float64; strict_dominance::Float64=0.1)
    # 1. Generate random sparse entries for the upper triangle
    # We generate two real sparse matrices for real/imag parts to control structure
    rng_real = sprand(n, n, density)
    rng_imag = sprand(n, n, density)

    # Combine to make complex weights.
    # We strictly mask to the upper triangle (k=1) to define unique edges first.
    U = triu(rng_real + im * rng_imag, 1)

    # 2. Make it Hermitian (A = A')
    # If U contains the upper triangle edges, A becomes the full adjacency matrix.
    A = U + U'

    # 3. Calculate Diagonal Requirements
    # For SDD, |M_ii| >= sum(|M_ij|) for all j != i.
    # We calculate the sum of absolute values of the off-diagonal row entries.
    row_sums = sum(abs, A, dims=2) # Returns an Nx1 matrix

    # 4. Construct the Matrix
    # We set M = D - A (Laplacian-like structure)
    # D must be at least 'row_sums'. We add 'strict_dominance' to ensure it is positive definite.

    # Create the sparse diagonal matrix
    D = spdiagm(0 => vec(row_sums) .+ strict_dominance)

    # M has positive diagonal and negative (complex) off-diagonals
    M = D - A

    return M
end

generate_random_sparse_hermitian_sdd (generic function with 1 method)

In [77]:
for i in 1:10
n = 1000
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
  fApxChol = apxCholMultiSDDSparse(S, 200) # Second param specifies the number of multiedges
  rel_eigs = eigvals(Matrix(S)-Matrix(fApxChol)*Matrix(fApxChol)')
  # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / maximum(abs.(eigvals(Matrix(S)))))
end

0.011508045148968412
0.010846123998698423
0.014918766381933663
0.010496799949854179
0.011298166976093846
0.01272115899893704
0.012120529986084295
0.01142927347655674
0.009682214219749605
0.010964654835421526


In [ ]:
for i in 1:10
n = 2000
  # FIX: density must be a positional argument, and strict_dominance should be a float
  S = generate_random_sparse_hermitian_sdd(n, 0.005; strict_dominance=0.000000001)
  fApxChol = apxCholMultiSDDSparse(S, 200) # Second param specifies the number of multiedges
  rel_eigs = eigvals(Matrix(S)-Matrix(fApxChol)*Matrix(fApxChol)')
  # The ratio of the smallest to the largest eigenvalue

  println(maximum(abs.(rel_eigs)) / maximum(abs.(eigvals(Matrix(S)))))
end

0.01564315716674432
0.018503148269035024
0.015649432238451767
